# Unit 2 Assignment: Mixture of Experts (MoE) Router

This notebook implements a **Smart Customer Support Router** using the Groq API.

## Goals
- Route incoming user messages into: `technical`, `billing`, or `general`
- Use a dedicated expert prompt for each category
- Return high-quality responses through a central orchestrator
- (Bonus) Route tool-style requests to a mock tool expert

## 1) Setup and Imports

In [7]:
# If needed, run this once:
# !pip install groq python-dotenv

import os
from typing import Dict
from dotenv import load_dotenv
from groq import Groq, BadRequestError

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

client = Groq(api_key=api_key)

# Prefer GROQ_MODEL from .env, then fall back to known active models.
_env_model = os.getenv("GROQ_MODEL", "").strip()
MODEL_CANDIDATES = [m for m in [
    _env_model,
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
] if m]

BASE_MODEL = MODEL_CANDIDATES[0]
print("Setup complete. Groq client initialized.")
print("Model candidates:", MODEL_CANDIDATES)

Setup complete. Groq client initialized.
Model candidates: ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']


## 2) Define Expert Configurations

In [2]:
MODEL_CONFIG: Dict[str, Dict[str, str]] = {
    "technical": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a Technical Support Expert. Be rigorous, precise, and code-focused. "
            "Diagnose root causes, provide clear steps, and include concise code examples when helpful."
        ),
    },
    "billing": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a Billing Support Expert. Be empathetic, policy-aware, and financially accurate. "
            "Explain charges, refunds, or subscriptions clearly and professionally."
        ),
    },
    "general": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a helpful General Assistant for casual queries and fallback support. "
            "Be concise, friendly, and practical."
        ),
    },
}

print("Experts loaded:", list(MODEL_CONFIG.keys()))

Experts loaded: ['technical', 'billing', 'general']


## 3) Router (Core Task)
The router returns **only one label**: `technical`, `billing`, `general`, or (bonus) `tool_use`.

In [8]:
ALLOWED_ROUTES = {"technical", "billing", "general", "tool_use"}

def _normalize_route(raw: str) -> str:
    route = (raw or "").strip().lower()
    return route if route in ALLOWED_ROUTES else "general"

def _safe_chat_completion(messages, temperature: float):
    last_error = None
    for model_name in MODEL_CANDIDATES:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                temperature=temperature,
                messages=messages,
            )
            return completion, model_name
        except BadRequestError as err:
            err_text = str(err).lower()
            if "decommissioned" in err_text or "no longer supported" in err_text or "model" in err_text:
                last_error = err
                continue
            raise
    raise RuntimeError(
        f"No usable model from MODEL_CANDIDATES={MODEL_CANDIDATES}. "
        "Set GROQ_MODEL in .env to a currently supported Groq model."
    ) from last_error

def route_prompt(user_input: str) -> str:
    router_system = (
        "You are an intent router for customer support.\n"
        "Classify the user message into exactly one category from:\n"
        "[technical, billing, general, tool_use].\n"
        "Use tool_use only when the user asks for real-time/external factual lookup (e.g., current price of Bitcoin).\n"
        "Return ONLY the category word. No punctuation, no explanation."
    )

    completion, _ = _safe_chat_completion(
        messages=[
            {"role": "system", "content": router_system},
            {"role": "user", "content": user_input},
        ],
        temperature=0,
    )

    raw_route = completion.choices[0].message.content
    return _normalize_route(raw_route)

## 4) Bonus Tool Expert (Mock)

In [4]:
def mock_get_bitcoin_price() -> str:
    # Mock tool response for assignment demo
    return "Bitcoin (BTC) current mock price: $64,250.00"

## 5) Orchestrator
`process_request(user_input)` routes the query, selects the expert, and returns final output.

In [9]:
def process_request(user_input: str) -> dict:
    route = route_prompt(user_input)

    if route == "tool_use":
        return {
            "route": route,
            "response": mock_get_bitcoin_price(),
            "model": "mock_tool",
        }

    expert = MODEL_CONFIG.get(route, MODEL_CONFIG["general"])

    completion, used_model = _safe_chat_completion(
        messages=[
            {"role": "system", "content": expert["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=0.7,
    )

    return {
        "route": route,
        "response": completion.choices[0].message.content,
        "model": used_model,
    }

## 6) Demo Runs

In [10]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey! Can you suggest a productivity tip for my day?",
    "What is the current price of Bitcoin?",
]

for q in test_queries:
    result = process_request(q)
    print("-" * 70)
    print("User:", q)
    print("Route:", result["route"])
    print("Model:", result["model"])
    print("Response:", result["response"])

----------------------------------------------------------------------
User: My python script is throwing an IndexError on line 5.
Route: technical
Model: llama-3.3-70b-versatile
Response: To accurately diagnose and resolve the `IndexError` in your Python script, I'll need more information. However, I can guide you through a general approach to troubleshooting this issue.

### Understanding IndexErrors

An `IndexError` in Python typically occurs when you try to access an element in a sequence (like a list, tuple, or string) using an index that is out of range. For example, if you have a list with 5 elements ( indexed from 0 to 4), trying to access the element at index 5 would result in an `IndexError`.

### Steps to Troubleshoot

1. **Identify the Line**: You've mentioned the error is on line 5. Review this line to understand what operation is being performed.

2. **Check the Sequence**: Ensure that the sequence (list, tuple, string, etc.) you're trying to access is indeed populated an

## 7) Summary

### What was implemented
- A Groq-powered MoE-style router with four routes: `technical`, `billing`, `general`, `tool_use`
- `route_prompt(user_input)` with deterministic routing (`temperature=0`)
- `process_request(user_input)` orchestrator that picks the right expert prompt
- Bonus mock tool function for Bitcoin price queries

